# Appendix A — Quantitative Toolbox (MM desk)

**Book:** Lehalle & Laruelle, *Market Microstructure in Practice* · App.A pp.248–315  
**Status:** `exp_run` · paper only · no orders  
**Out:** `research/books/mmip/out/appendix_quant/`

## Executive takeaway

- **HL is tick-constrained** (~99% 1-tick); **Lighter is continuous** (~15% 1-tick, mean ε≈11.5).
- **E-min schedules** on real HL ETH \(V_n\) deviate from uniform by mean L1≈**0.40**.
- **Xvenue Epps** (HL mid vs Deribit ETH): corr≈0 at ≤60s → **~0.87 at 600s**.
- Trade arrivals **cluster** (CV≫1; lag-1 ACF 0.17–0.44). Hawkes MoM \(R\) is indicative only.
- FEI: link to Ch.1 only — do not redo spatial fragmentation here.


## Theory / formulas

| § | Formula | Code |
|---|---------|------|
| A.5 | \(\varepsilon=(A-B)/\tau\), leeway \(\varepsilon-1\) | `harris_tick_features` |
| A.6 | \(v_n \propto V_n/\sigma_n^{1/\gamma}\) | `schedule_expectation_min` |
| A.6 | \(C=\mathbb{E}(W)+\lambda\mathrm{Var}(W)\) | `schedule_mean_variance_linear` |
| A.11 | \(\lambda(t)=\lambda_0+\sum \alpha e^{-\beta(t-\tau_i)}\) | `hawkes_descriptive` (MoM) |
| A.12 | \(\widehat{\mathrm{corr}}(\Delta)\) last-tick sync | `epps_correlation` |
| A.1 | \(F=H(q)/\log N\) | Ch.1 + `fei` |

**Honesty:** schedule κ/λ uncalibrated; Hawkes ≠ MLE; xasset Epps on 1m marks is coarse.


In [ ]:
from pathlib import Path
import json
BOOK_ROOT = Path('../..').resolve()
OUT = BOOK_ROOT / 'out' / 'appendix_quant'
payload = json.loads((OUT / 'exp_appa_summary.json').read_text())
print('created', payload['created_at'])
print('symbol', payload['symbol'], 'tob_day', payload['tob_day'])
print('warehouse_days', payload['warehouse_days'])


## Data & method

1. Collector TOB `20260929` ETH → Harris  
2. HL trade tape DENSE days → volume curve + Hawkes  
3. Deribit 1m marks ETH/BTC → xasset Epps / signature  
4. HL `l2_rebuild` vs Deribit mark (2026-09-26) → xvenue Epps  
5. Ch.1 JSON → FEI link only  

No ClickHouse MCP.


In [ ]:
import pandas as pd
h = payload['harris']['venues']
rows = []
for v, m in h.items():
    rows.append({
        'venue': v, 'tick': m['tick'], 'frac_one_tick': m['frac_one_tick'],
        'mean_eps_ticks': m['mean_spread_ticks'], 'mean_leeway': m['mean_leeway'],
        'mean_spread_bps': m['mean_spread_bps'], 'rel_tick_bps': m['mean_rel_tick_bps'], 'n': m['n'],
    })
pd.DataFrame(rows)


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / 'fig_harris_tick.png')))


## A.6 Schedule toys on real \(V_n\)

Expectation-min (Prop.1) vs uniform vs mean–var toy. σ from within-hour trade log-increments (proxy).


In [ ]:
sch = payload['schedule']
sched_days = pd.DataFrame(payload['schedule_days_summary'])
print('mean L1(E-min vs uniform) =', sch['mean_l1_exp_vs_uni'])
sched_days


In [ ]:
display(Image(filename=str(OUT / 'fig_schedule.png')))


## A.12 Epps / signature

Xasset on 1m marks shows little classic decay (already synchronous). **Xvenue** HL↔Deribit shows the textbook rise of correlation with Δ.


In [ ]:
epps_xa = pd.DataFrame(payload['epps_hawkes']['epps_xasset_eth_btc_deribit'])
epps_xv = pd.DataFrame(payload['epps_hawkes']['epps_xvenue_hl_deribit_eth'])
print('xasset ETH-BTC (Deribit marks)')
display(epps_xa)
print('xvenue HL mid vs Deribit ETH mark', payload['epps_hawkes']['epps_xvenue_day'])
display(epps_xv)


In [ ]:
display(Image(filename=str(OUT / 'fig_epps_xasset.png')))
display(Image(filename=str(OUT / 'fig_epps_xvenue.png')))


## A.11 Hawkes-style clustering (descriptive)

Inter-arrival CV ≫ 1 and positive count ACF ⇒ clustering. MoM branching ratio is **Hold** (not MLE).


In [ ]:
hawkes = pd.DataFrame(payload['epps_hawkes']['hawkes_hl_eth'])
hawkes['R_mom'] = hawkes['hawkes_mom'].apply(lambda x: (x or {}).get('branching_ratio'))
hawkes[['day','n_events','mean_rate_per_s','interarrival_cv','acf_lag1','acf_lag5','R_mom']]


In [ ]:
display(Image(filename=str(OUT / 'fig_hawkes_acf.png')))


## A.1 FEI link to Ch.1

Do not redo Ch.1. Book Table 1.2 benchmarks + Ch.1 observed FEI.


In [ ]:
fei = payload['fei_link']
print('Book Table 1.2:', fei['book_table_1_2'])
print('Ch.1 FEI size / updates:', fei.get('ch01_fei_size'), fei.get('ch01_fei_updates'))
print('Ch.1 size share:', fei.get('ch01_size_share'))
print('Trade hourly FEI (temporal):', fei.get('ch01_trade_hourly_fei_mean'))
print(fei.get('honesty'))


## MM interpretation

| Function | Action |
|----------|--------|
| Quoting | HL: size/skew/cancel; Lit: continuous tick management |
| POV / schedule | Child weights ∝ \(V_n/\sigma_n\); avoid flat TWAP prior |
| Xvenue hedge | Sync horizon ≥ minutes (Epps) before trusting corr |
| Intensity | Elevated ACF/CV → burst gate research |
| Fragmentation | Use Ch.1 Promote list (`crossed_nbbo`, `update_share`, tick) |

## Promote / Hold / Kill

See `CANDIDATES.md`.

**Promote:** `tick.frac_one_tick`, `tick.spread_leeway`, `tick.rel_tick_bps`, `sched.vol_curve_share`, `sched.expectation_min`, `epps.xvenue_corr`, `hawkes.count_acf`.

**Hold:** Harris MLE, mean–var λ, xasset Epps on 1m marks, Hawkes MoM \(R\), FEI spatial trade.

**Kill:** SOR ODE toy, flash-crash toy.
